In [1]:
! pip3 install transformers

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import torch
from transformers import pipeline


In [3]:
device = "mps" if torch.backends.mps.is_available() else ("cuda:0" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device == "mps" else torch.float32

In [4]:
ask_llm = pipeline(
  task="text-generation",
  model="Qwen/Qwen2.5-3B-Instruct",
  device=device,
  torch_dtype=dtype
)

print(ask_llm("Who is Scott Lai?")[0]["generated_text"])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Who is Scott Lai? Scott Lai is a Taiwanese-American entrepreneur, investor, and angel investor. He is the founder of Vela Capital, a venture capital firm that invests in early-stage technology companies. Prior to founding Vela Capital, Scott was a partner at Andreessen Horowitz, one of the world's leading venture capital firms.

Scott Lai has made significant contributions to the tech industry through his investments and mentorship. He is known for his expertise in artificial intelligence, cybersecurity, and blockchain technologies. Some of the notable companies he has invested in include:

1. Cognitivescale
2. Cylance
3. Datadog
4. Looker
5. MuleSoft
6. Sift Science
7. SigOpt
8. Tala

In addition to his work with Vela Capital and Andreessen Horowitz, Scott Lai has also been involved in various other ventures and initiatives. He is a frequent speaker at tech conferences and events, sharing insights on emerging trends and investment opportunities in the tech industry.

As an angel inves

As you can see here, the model has no idea who I am from above response.

Let's cook it!

First, let's teach the model who I am. Here you can use your personal data to generate the exact format you will use for fine-turning base on your own data. You can use ChatGPT for this, just ask it to transfer your resume into the trainable json format with "prompt" and "completion"

In [5]:
! pip install datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [6]:
# load data 
from datasets import load_dataset

raw_data = load_dataset('json', data_files = "scott_lai_resume_train.json")
raw_data

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 122
    })
})

In [7]:
raw_data["train"][0]

{'prompt': 'What is Scott Lai’s profession?',
 'completion': 'AI Engineer and Data Scientist.'}

As you can see, here we return with the long text, but for fine-tuning we need the data to be small and precise chunks, more like here we apply the tokenization to take the text and split it into smaller chunks. Each chunk is called a token and it the smallest unit of meaning that LLMs work with.

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)
def preprocess(sample):
    sample = sample['prompt']+ '\n' + sample['completion']
    print(sample)
    tokenized = tokenizer(
        sample,
        max_length = 128,
        truncation = True,
        padding = "max_length"    
    )

    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized
data = raw_data.map(preprocess)


Map:   0%|          | 0/122 [00:00<?, ? examples/s]

What is Scott Lai’s profession?
AI Engineer and Data Scientist.
How many years of experience does Scott Lai have in generative AI and LLM solutions?
Over 5 years.
What infrastructures is Scott Lai skilled in designing and optimizing?
Scalable ML infrastructures using PyTorch, Hugging Face, and FastAPI on AWS.
What type of workflows is Scott Lai experienced in building?
End-to-end pipelines, scalable microservices, and ETL workflows.
What collaboration experience does Scott Lai have?
Proven track record in cross-functional collaboration and implementing ML and data engineering best practices.
Which skill in Programming & Scripting does Scott Lai have?
Python
Which skill in Programming & Scripting does Scott Lai have?
Rust
Which skill in Programming & Scripting does Scott Lai have?
Node.js
Which skill in Programming & Scripting does Scott Lai have?
HTML
Which skill in Programming & Scripting does Scott Lai have?
CSS
Which skill in Programming & Scripting does Scott Lai have?
JavaScript
W

In [10]:
print(data['train'])

Dataset({
    features: ['prompt', 'completion', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 122
})


## LoRA

now, let's move into the training

In [11]:
! pip install peft

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [12]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map = device,
    torch_dtype = torch.float16
)

lora_config = LoraConfig (
    
    task_type = TaskType.CAUSAL_LM, 
    target_modules=['q_proj', "k_proj", "v_proj"]
)
model = get_peft_model(model, lora_config)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
from transformers import TrainingArguments, Trainer


train_args = TrainingArguments(
    num_train_epochs = 10, # we will go throught the dataset from start to finish 10 times
    learning_rate=0.001, 
    logging_steps = 25, # we want to see the result in every 25 steps it runs 
    fp16 = False # float point set to 16 to speed it up, set to "True" if you are on GPU
)

trainer = Trainer(
    args = train_args,
    model = model, 
    train_dataset=data["train"]
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [15]:
trainer.train()

Step,Training Loss
25,2.382600
50,0.474900
75,0.265800
100,0.207800
125,0.174200
150,0.143400


TrainOutput(global_step=160, training_loss=0.5786425687372685, metrics={'train_runtime': 40.5857, 'train_samples_per_second': 30.06, 'train_steps_per_second': 3.942, 'total_flos': 2602200748523520.0, 'train_loss': 0.5786425687372685, 'epoch': 10.0})

In [16]:
# save the model
trainer.save_model("./my-qwen")
tokenizer.save_pretrained("./my-qwen")

('./my-qwen/tokenizer_config.json',
 './my-qwen/special_tokens_map.json',
 './my-qwen/chat_template.jinja',
 './my-qwen/vocab.json',
 './my-qwen/merges.txt',
 './my-qwen/added_tokens.json',
 './my-qwen/tokenizer.json')

Now let's test it out

In [17]:
ask_llm = pipeline(
  task="text-generation",
  model="./my-qwen",
  tokenizer='./my-qwen',
  device=device,
  torch_dtype=dtype
)

print(ask_llm("Who is Scott Lai?")[0]["generated_text"])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Who is Scott Lai? 
A professional UI/UX designer with 5+ years of experience.
